# TF-IDF と線形モデルによる文書分類ベースライン・エラー分析

書籍 第1章（1-5「自然言語処理システムの開発サイクル」）および第2章（2-1・2-2 文書分類）に対応する演習ノートブックです。
[BERT による文書分類](https://colab.research.google.com/github/mamorlis/nlpbook/blob/main/notebooks/ch2_bert.ipynb)と同じデータセット（`covid-annotation.csv`）を使い、CPU のみ・数秒で以下を体験します。

1. **軽量ベースラインの構築**: TF-IDF + ロジスティック回帰で訓練し、正解率と F1 を測る
2. **特徴量重みの可視化**: モデルが `TRUE` / `FALSE` の判断にどの表現を使っているかを確認する
3. **エラー分析**: 誤分類された文書を `コメント` 列と照らし合わせ、線形モデルの限界（否定・主語の取り違え等）を分析する

## 1. データの準備と分割

第2章の BERT 演習と同じ `covid-annotation.csv`（1,000件：`TRUE` 500件 / `FALSE` 500件）をダウンロードし、各集合でラベルの比率を保つように層化分割します。まず全体の10%をテスト用に取り分け、残りの10%を開発用にするので、訓練810件・開発90件・テスト100件になります。

BERT 演習も同じ件数とシードを使いますが、分割に使うライブラリが違うため、同じ事例が各集合に入る保証はありません。モデル間の性能差を測るには、訓練・開発・テストの行 ID を両モデルで共通にしてください。

In [1]:
!curl -fLO https://raw.githubusercontent.com/mamorlis/nlpbook/refs/heads/main/misc/covid-annotation.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

100  532k    0  532k    0     0  2213k      0 --:--:-- --:--:-- --:--:-- 2220k


In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

df = pd.read_csv("covid-annotation.csv")
df["y"] = df["ラベル"].astype(int)

# 全体の10%をテストデータ、残りの10%を開発データ、それ以外を訓練データにする
train_dev_df, test_df = train_test_split(df, test_size=0.1, stratify=df["y"], random_state=42)
train_df, dev_df = train_test_split(train_dev_df, test_size=0.1, stratify=train_dev_df["y"], random_state=42)

print(f"訓練: {len(train_df)} 件, 開発: {len(dev_df)} 件, テスト: {len(test_df)} 件")

訓練: 810 件, 開発: 90 件, テスト: 100 件


---

## 2. TF-IDF + ロジスティック回帰によるベースライン学習

日本語の形態素解析器を入れなくても動作するよう、文字 n-gram（1〜3文字）の TF-IDF を特徴量としてロジスティック回帰を学習します。

In [3]:
clf = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3), min_df=2),
    LogisticRegression(max_iter=1000, C=3.0, random_state=42),
)
clf.fit(train_df["文書"], train_df["y"])

for name, sub in [("開発データ", dev_df), ("テストデータ", test_df)]:
    pred = clf.predict(sub["文書"])
    acc = accuracy_score(sub["y"], pred)
    f1 = f1_score(sub["y"], pred)
    print(f"{name:8s} | Accuracy = {acc:.4f}, F1 = {f1:.4f}")

print("\nテストデータの詳細:")
print(classification_report(test_df["y"], clf.predict(test_df["文書"]), target_names=["FALSE", "TRUE"], digits=4))

開発データ    | Accuracy = 0.9889, F1 = 0.9888
テストデータ   | Accuracy = 0.9600, F1 = 0.9592

テストデータの詳細:


              precision    recall  f1-score   support

       FALSE     0.9423    0.9800    0.9608        50
        TRUE     0.9792    0.9400    0.9592        50

    accuracy                         0.9600       100
   macro avg     0.9607    0.9600    0.9600       100
weighted avg     0.9607    0.9600    0.9600       100



CPU のみ・1秒未満で高いベースライン性能が出ることがわかります。深層学習モデル（BERT）を使う前に、こうした軽量ベースラインでデータの傾向や妥当性を素早く確認するのが実務では有効です（書籍 1-5、2-2）。

---

## 3. 予測に効いている特徴量（重み）の可視化

重み係数の符号を見ると、その文字 n-gram が `TRUE` と `FALSE` のどちらに予測を傾けるかが分かります。個々の文書での寄与は、重み係数と、その文書での TF-IDF 値の積です。

In [4]:
vec = clf.named_steps["tfidfvectorizer"]
lr = clf.named_steps["logisticregression"]
feature_names = np.array(vec.get_feature_names_out())
coefs = lr.coef_[0]

top_pos = np.argsort(coefs)[::-1][:15]
top_neg = np.argsort(coefs)[:15]

pd.DataFrame({
    "TRUE に寄与する表現": feature_names[top_pos],
    "重み(+)": coefs[top_pos].round(3),
    "FALSE に寄与する表現": feature_names[top_neg],
    "重み(-)": coefs[top_neg].round(3),
})

,TRUE に寄与する表現,重み(+),FALSE に寄与する表現,重み(-)
0,の,1.765,」,-1.447
1,染して,1.717,「,-1.447
2,わか,1.675,出た,-1.382
3,がわ,1.574,が出た,-1.382
4,がわか,1.574,性とな,-1.118
5,か,1.512,われま,-1.050
6,かりま,1.474,検,-1.017
7,わかり,1.442,検査,-1.017
8,とがわ,1.442,で陽性,-1.017
9,染し,1.378,で陽,-1.017


---

## 4. 誤分類事例のエラー分析

テストデータでモデルが誤分類した事例と、`covid-annotation.csv` に記録されている `コメント`（ラベル付けの根拠）を並べて確認してみましょう。

この演習では最終モデルの挙動を理解するために、テストデータの誤分類を確認します。結果をもとにモデルを調整する場合は、開発データでエラー分析を行い、最終評価用のデータを別に確保します。

In [5]:
test_df = test_df.copy()
test_df["予測"] = clf.predict(test_df["文書"])
test_df["予測確率(TRUE)"] = clf.predict_proba(test_df["文書"])[:, 1].round(3)

errors = test_df[test_df["y"] != test_df["予測"]]
print(f"テストデータの誤分類件数: {len(errors)} / {len(test_df)} 件\n")

for _, row in errors.head(8).iterrows():
    gold_lbl = "TRUE" if row["y"] == 1 else "FALSE"
    pred_lbl = "TRUE" if row["予測"] == 1 else "FALSE"
    print(f"[ID {row['文書ID']}] 正解={gold_lbl} / 予測={pred_lbl} (p={row['予測確率(TRUE)']}) | コメント: {row['コメント']}")
    print(f"  文書: {row['文書'][:120]}...")
    print()

テストデータの誤分類件数: 4 / 100 件

[ID 47] 正解=TRUE / 予測=FALSE (p=0.423) | コメント: アノテーション元データのラベル準拠
  文書: 今月7日に体調が急変して入院した際には肺炎と診断されました。 先週の検査ではウイルス性肺炎であることが判明しました。...

[ID 37] 正解=TRUE / 予測=FALSE (p=0.46) | コメント: コロナウイルスによる肺炎に罹患
  文書: 今回のコロナウイルスによる肺炎という病気は 私には初めてですので、どんな治療法があるのかも分かりません。 良くなったらまたブログやInstagramなどでもお知らせしますね!...

[ID 32] 正解=FALSE / 予測=TRUE (p=0.64) | コメント: 簡易検査陽性の後のPCR検査で陰性
  文書: 先日、検査結果から新型コロナウイルス陽性反応が出たため、 病院へ行きました。 そこでPCR検査を受けた結果、陰性だったことがわかりました。...

[ID 16] 正解=TRUE / 予測=FALSE (p=0.452) | コメント: 検査でコロナ感染判明
  文書: 私は熱が出て病院で検査を受けました。 そして、結果は新型コロナウイルスに感染したというものでした。 私は今までインフルエンザ以外には何もかも引いたことがない人だから、すごくショックです。...



### まとめ

- TF-IDF + ロジスティック回帰でも全体の傾向はよく捉えられますが、「感染した**わけではない**（否定）」「**友人・同僚**が感染した（主語の違い）」「**もし**感染したら（仮定）」のような文脈や構文に依存する事例では誤りやすくなります。
- こうしたエラー分析を通じて「文脈を考慮できる事前学習モデル（BERT）を導入する根拠」を明確にすることができます。